In [39]:
!pip install optuna


In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from numpy.lib.stride_tricks import sliding_window_view
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import weight_norm
from pytorch_tcn import TCN
from numpy import mean
import matplotlib.pyplot as plt
from skopt.space import Integer, Real, Categorical
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import optuna

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### TCN

#### Data Prep

In [41]:
training_data  = pd.read_csv("final_train_data.csv")
val_data = pd.read_csv("val_data.csv.gz")
test_data = pd.read_csv("test_data.csv.gz")

In [42]:
training_data.head()

,Distance (cm),Illuminance (lx),amplitude,frequency,Magnetic field x (µT),Magnetic field y (µT),Magnetic field z (µT),Acceleration x (m/s^2),Acceleration y (m/s^2),Acceleration z (m/s^2),...,Linear Acceleration x (m/s^2),Linear Acceleration y (m/s^2),Linear Acceleration z (m/s^2),Common time (s),Activity,Mood,Arousal,Social engagement,Noise Level,Concentration Level
0,0.444559,67.119613,0.003641,3.964895e-19,18.961148,-71.217642,77.025529,-1.465748,5.065817,5.083255,...,0.061589,0.029563,0.080543,0.001231,rest,3,3,1,2,1
1,0.555727,67.184122,0.004651,3.616358e-20,19.588866,-73.267046,79.667438,-1.588174,5.482184,5.694939,...,0.063213,0.030341,0.082709,0.002898,rest,3,3,1,2,1
2,0.666895,67.248631,0.005686,3.298460e-21,19.766805,-73.902556,80.417662,-1.710600,5.898552,6.306622,...,0.063745,0.030597,0.083409,0.005110,rest,3,3,1,2,1
3,0.778062,67.313140,0.006703,3.008506e-22,19.817245,-74.099625,80.630704,-1.833026,6.314919,6.918306,...,0.063919,0.030680,0.083634,0.007505,rest,3,3,1,2,1
4,0.889230,67.377649,0.007591,2.744041e-23,19.831543,-74.160735,80.691202,-1.877866,6.460432,7.147014,...,0.063976,0.030708,0.083707,0.009961,rest,3,3,1,2,1


In [43]:
#training data
y = training_data["Activity"]
X = training_data.drop("Activity",axis = 1)

#validation data
y_val = val_data["Activity"]
X_val = val_data.drop("Activity",axis = 1)

#test data
y_test = test_data["Activity"]
X_test = test_data.drop("Activity",axis = 1)

In [44]:
#encoding of lavels to numeric
le  = LabelEncoder()
y_transformed = le.fit_transform(y)
y_val_transformed = le.transform(y_val)
y_test_transformed = le.transform(y_test)

In [45]:
def windowing(X,y_transformed, window_size = 500):    
    """function that applies windowing so that training is better"""
    step_size = window_size//2
    X_windows = []
    y_windows = []

    for start in range(0, len(X)-window_size +1, step_size):
        end = start+window_size
        X_window = X[start:end]
        y_window = y_transformed[start:end]
        X_windows.append(X_window)
        y_windows.append(y_window)
    return X_windows, y_windows

#windowing for training
X_windows,y_windows = windowing(X,y_transformed)
#windowing for val
X_val_windows,y_val_windows = windowing(X_val,y_val_transformed)
#windowing for test
X_test_windows,y_test_windows = windowing(X_test,y_test_transformed)

In [46]:
#transformation to tensors
#training
X = torch.tensor(np.array(X_windows), dtype=torch.float32)
y = torch.tensor(np.array(y_windows),dtype = torch.long)
#validation 
X_val = torch.tensor(np.array(X_val_windows), dtype=torch.float32)
y_val = torch.tensor(np.array(y_val_windows),dtype = torch.long)
#testing
X_test = torch.tensor(np.array(X_test_windows), dtype=torch.float32)
y_test = torch.tensor(np.array(y_test_windows),dtype = torch.long)

In [47]:
#putting everything into datalodaer
#train
dataset =TensorDataset(X,y)
loader = DataLoader(dataset, batch_size=32, shuffle=False) #shuffle ==false since it is sequential
#val
val_dataset =TensorDataset(X_val,y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
#val
test_dataset =TensorDataset(X_test,y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [48]:
#adapted from https://github.com/locuslab/TCN/blob/master/TCN/tcn.py

# class chomping(nn.Module):
#     def __init__(self, padding_size):
#         self.padding_size = padding_size
#     def forward()
# class Temporal_Block(nn.Module):
#     def __init__(self,num_classes,dilation,kernel_size):
#         super(Temporal_Block,self).__init__()
#         self.conv_layer1 = nn.Conv1d()


In [ ]:
#initializing TCN
num_channels = np.array([32,64,128])
kernel_size = 3
dilation_reset =16
dropout = 0.1
causal = True
use_norm = "weight_norm"
activation = "relu"
kernel_initializer = "xavier_uniform"
use_gate = False
use_skip_connections = True

tcn_model = TCN(num_inputs=X.shape[2], num_channels = num_channels, kernel_size = kernel_size,
                dilation_reset = dilation_reset, causal =causal, use_norm = use_norm,
                activation =activation,kernel_initializer = kernel_initializer, 
                use_skip_connections =use_skip_connections,embedding_shapes =None,use_gate =use_gate,
                 lookahead = 0, output_projection  =6, output_activation = None ).to(device)

In [50]:
# initializing loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tcn_model.parameters(),lr = 0.001)

In [ ]:
# training function
def training_tcn(tcn_model,n_epochs,loader=loader,optimizer = optimizer, loss_fn = loss_fn, training=False):
    """training function for our TCN model"""
    training_acc = []
    training_loss = []

    model = tcn_model
    for epoch in range(n_epochs):
        epoch_loss = []
        epoch_accuracy = []
        if training:
            print(f"this is epoch {epoch}")
    #loops over epochs and batches
        for batch,data in enumerate(loader):

            input,label =data #defines input and labels per batch
            input = input.to(device)
            label = label.to(device)
            
            input = input.permute(0, 2, 1) #permute to fit expected shape (batch/features/window)

            #set optimizer to zero 
            optimizer.zero_grad()
            outputs = model(input)
            pred = torch.argmax(outputs,axis = 1 )
            #loss
            loss = loss_fn(outputs,label)
            loss.backward()  #calculates loss with backprop
            #adjust learning weights
            optimizer.step()
            epoch_loss.append(loss.item())
            epoch_accuracy.append((pred==label).sum().item()/label.numel())
        training_acc.append(mean(epoch_accuracy))
        training_loss.append(mean(epoch_loss))
        if training:
            print(f"for this eopich the mean loss was {mean(epoch_loss)}")    
            print(f"for this eopich the mean accuracy  was {mean(epoch_accuracy)}") 
    if training:
        # Plot the loss metrics
        plt.plot(training_loss)
        plt.xlabel("steps")
        plt.ylabel("loss")
        plt.ylim(0)
        plt.show()






In [ ]:
def eval_function(tcn_model,test_loader,loss_fn, test_mode=False):
    """function that evaluates for test and hp tuning"""
    model = tcn_model
    model.eval()
    test_loss = []
    test_acc=[]
    test_preds =[]
    test_labels = []
    for batch,data in enumerate(test_loader):
        input,label =data
        input = input.to(device)
        label = label.to(device)
        input = input.permute(0, 2, 1)
        pred_log = model(input)
        pred = torch.argmax(pred_log,axis=1)
        loss = loss_fn(pred_log,label)
        test_loss.append(loss.item())
        test_acc.append((pred==label).sum().item()/label.numel())
        test_preds.extend(pred.cpu().numpy())
        test_labels.extend(label.cpu().numpy())
    f1 = f1_score(test_preds,test_labels)
    
    if test_mode:
        print(classification_report(test_labels,test_preds))
        confusion = confusion_matrix(test_labels,test_preds)
        dis = ConfusionMatrixDisplay(confusion_matrix=confusion,display_labels=le.classes_)
        dis.plot(cmap="Greens")
        return f1, mean(test_loss),mean(test_acc)
    else:
        return f1

    

In [ ]:
def objective_fn(trial):
    #hp space
    num_channels = trial.suggest_categorical("num_channels", [[32, 64, 128], [16, 32, 64], [64, 64, 64]])
    kernel_size = trial.suggest_int("kernel_size", 2, 7)
    dilation_reset = trial.suggest_categorical("dilation_reset", [8, 16, 32])
    use_norm = trial.suggest_categorical("use_norm", ["weight_norm", "batch_norm", None])
    activation = trial.suggest_categorical("activation", ["relu", "tanh", "gelu"])
    kernel_initializer = trial.suggest_categorical("kernel_initializer", ["xavier_uniform", "kaiming_uniform", "orthogonal"])
    use_gate = trial.suggest_categorical("use_gate", [True, False])
    use_skip_connections = trial.suggest_categorical("use_skip_connections", [True, False])

    tcn_model = TCN(num_inputs=X.shape[2], num_channels = num_channels, kernel_size = kernel_size,
                dilation_reset = dilation_reset, causal =causal, use_norm = use_norm,
                activation =activation,kernel_initializer = kernel_initializer, 
                use_skip_connections =use_skip_connections,embedding_shapes =None,use_gate =use_gate,
                 lookahead = 0, output_projection  =6, output_activation = None ).to(device)
    
    #loss fn etc
    optimizer = torch.optim.Adam(tcn_model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    #training
    training_tcn(tcn_model,n_epochs=10)
    f1 = eval_function(tcn_model,test_loader,loss_fn, test_mode=False)
    return f1
    

In [ ]:
study = optuna.create_study(direction="maximize")  
study.optimize(objective_fn, n_trials=50, show_progress_bar=True, gc_after_trial=True)
best_params = study.best_params
print("Best hyperparameters:", best_params)

[I 2025-06-19 23:29:38,631] A new study created in memory with name: no-name-182720eb-96cf-4f6f-9381-5c3228506c1f


  0%|          | 0/50 [00:00<?, ?it/s]

/Users/christophlaute/Data Mining Project/Data-Mining-Project-1/.conda/lib/python3.12/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [32, 64, 128] which is of type list.
  warnings.warn(message)
/Users/christophlaute/Data Mining Project/Data-Mining-Project-1/.conda/lib/python3.12/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [16, 32, 64] which is of type list.
  warnings.warn(message)
/Users/christophlaute/Data Mining Project/Data-Mining-Project-1/.conda/lib/python3.12/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [64, 64, 64] which is of type list.
  warnings.warn(message)


this is epoch 0
this is epoch 1
[W 2025-06-19 23:29:52,746] Trial 0 failed with parameters: {'num_channels': [16, 32, 64], 'kernel_size': 5, 'dilation_reset': 32, 'use_norm': None, 'activation': 'tanh', 'kernel_initializer': 'xavier_uniform', 'use_gate': True, 'use_skip_connections': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/christophlaute/Data Mining Project/Data-Mining-Project-1/.conda/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/_9/q20vv8ss26d48h9872glly8h0000gn/T/ipykernel_44551/3724494454.py", line 22, in objective_fn
    training_tcn(tcn_model,n_epochs=10)
  File "/var/folders/_9/q20vv8ss26d48h9872glly8h0000gn/T/ipykernel_44551/4262383221.py", line 19, in training_tcn
    outputs = model(input)
              ^^^^^^^^^^^^
  File "/Users/christophlaute/Data Mining Project/Data-Mining-Project-1

KeyboardInterrupt: 

In [ ]:
#training_tcn(tcn_model,n_epochs=10, training = True)